# StratPPI example

Fisch et al., NeurIPS 2024. Data and split -> fixed strata -> target -> inference -> output. Run from `examples/simulation/`. The fixed strata are x0 < 0 and x0 >= 0; each has known probability 0.5 under this generator.


In [1]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

from openppi import (Mean, PPIData, StratifiedPPIData, get_method, split,
                     stratified_allocate, confidence_residual_variance)
from dataset.judge_sim import make_judge_data


In [2]:
# 1. Fit only on the independent training sample, then fix the model.
frame = make_judge_data(n=2000, seed=0)
train, labeled, unlabeled = split(
    frame.drop(columns='label').to_numpy(), frame['label'].to_numpy(),
    train_size=400, labeled_size=400, random_state=42)
model = RandomForestRegressor(n_estimators=60, min_samples_leaf=8,
                              max_depth=8, random_state=0, n_jobs=1).fit(*train)
f_labeled = model.predict(labeled[0])
f_unlabeled = model.predict(unlabeled)


In [3]:
# 2. Fixed feature-based strata; weights come from the population definition.
strata = {}
for key, negative in [('negative_x0', True), ('nonnegative_x0', False)]:
    mask_l = (labeled[0][:, 0] < 0) == negative
    mask_u = (unlabeled[:, 0] < 0) == negative
    strata[key] = PPIData(labeled[1][mask_l], f_labeled[mask_l], f_unlabeled[mask_u])
weights = {'negative_x0': .5, 'nonnegative_x0': .5}
data = StratifiedPPIData(strata=strata, weights=weights)
target = Mean()


In [4]:
# 3. Tune prediction weights separately within strata; form one Wald interval.
result = get_method('StratPPI').inference(data, target=target, alpha=.05)

# 4. Output.
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
for key, info in result.diagnostics['strata'].items():
    print(key, 'lambda:', info['prediction_weight'],
          'labeled:', info['labeled_count'], 'unlabeled:', info['unlabeled_count'])


Estimate: 0.4978656464000275
Confidence interval: (0.45138886360233005, 0.5443424291977249)
negative_x0 lambda: 0.5049278973030292 labeled: 199 unlabeled: 603
nonnegative_x0 lambda: 0.6295869676633844 labeled: 201 unlabeled: 597


## Optional: plan a fresh stratified experiment

Use an independent unlabeled pilot and a fixed planning lambda of 1. Treat model predictions as approximate confidence scores. The plan depends on no inference labels. These counts are for a future sample; they do not retroactively change the experiment above.


In [5]:
pilot = make_judge_data(n=2000, seed=123).drop(columns='label').to_numpy()
f_pilot = model.predict(pilot)
residual_variances, prediction_variances = {}, {}
for key, negative in [('negative_x0', True), ('nonnegative_x0', False)]:
    f = f_pilot[(pilot[:, 0] < 0) == negative]
    residual_variances[key] = confidence_residual_variance(f, f, lam=1.)
    prediction_variances[key] = float(np.var(f, ddof=1))
plan = stratified_allocate(weights, residual_variances, prediction_variances,
                           labeled_budget=400, unlabeled_budget=1200)
print('Planned labeled counts:', plan.labeled_counts)
print('Planned unlabeled counts:', plan.unlabeled_counts)


Planned labeled counts: {'negative_x0': 205, 'nonnegative_x0': 195}
Planned unlabeled counts: {'negative_x0': 564, 'nonnegative_x0': 636}
